# Hybrid search 

example with information extracted from png

### 1. PNGs -> JSON

In [ ]:
from pathlib import Path

api_key = 'REDACTED_SEE_ENV'

out_dir = Path("data")

out_dir.mkdir(exist_ok=True)

MODELS = [
    # {"name": "gemini-2.0-flash", "input_m": 0.10, "output_m": 0.40}
    {"name": "gemini-2.5-pro",              "input_m": 1.25,  "output_m": 10.00,  "note": "<=200k tokens"}
]


IMAGE_PATHS = [
    "images/newey1.png", 
    "images/newey2.png", 
    "images/newey3.png",
    "images/newey4.png",
    "images/newey5.png",
    "images/newey6.png",
    "images/newey7.png",
    "images/newey8.png",
    "images/newey9.png",
    "images/newey10.png",
    "images/newey11.png",
]


In [ ]:
# %pip install tqdm pandas matplotlib pyyaml

In [ ]:
import requests
import json
import time
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm # Sleek Jupyter progress bars


with open("prompts.yaml", "r") as f:
    PROMPTS = yaml.safe_load(f)["prompts"]

out_dir.mkdir(exist_ok=True)
all_results = []

def process_task(img_path, file_uri, model_info, p_data):
    model_name = model_info["name"]
    p_id = p_data["id"]
    start_time = time.time()
    
    payload = {
        "system_instruction": {"parts": [{"text": p_data["system"]}]},
        "contents": [{"role": "user", "parts": [
            {"text": p_data["user"]},
            {"file_data": {"mime_type": "image/png", "file_uri": file_uri}}
        ]}],
        "generationConfig": {"temperature": 0.1}
    }

    try:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
        r = requests.post(url, params={"key": api_key}, json=payload, timeout=120)
        r.raise_for_status()
        data = r.json()

        # Extract the response text
        response_text = data["candidates"][0]["content"]["parts"][0]["text"]
        
        # Extract usage and calculate cost
        usage = data.get("usageMetadata", {})
        p_t = usage.get("promptTokenCount", 0)
        c_t = usage.get("candidatesTokenCount", 0)
        cost = (p_t / 1e6 * model_info["input_m"]) + (c_t / 1e6 * model_info["output_m"])

        # NEW: Include all fields required for consolidation and dashboard
        res = {
            "image": Path(img_path).name,
            "model": model_name,
            "prompt_id": p_id,
            "response": response_text,
            "usage": {
                "prompt": p_t,
                "completion": c_t,
                "cost_usd": round(cost, 6)
            },
            "time_s": time.time() - start_time
        }
        
        # Save JSON
        filename = f"{model_name.replace('.', '_')}_{Path(img_path).stem}_{p_id}.json"
        (out_dir / filename).write_text(json.dumps(res, indent=2))
        return res
    except Exception as e:
        return {"error": str(e), "image": Path(img_path).name, "prompt_id": p_id}

# --- Execution ---
# 1. Sequential Upload
file_uris = {}
for img_path in tqdm(IMAGE_PATHS, desc="Uploading Images"):
    upload_url = f"https://generativelanguage.googleapis.com/upload/v1beta/files?key={api_key}"
    with open(img_path, "rb") as f:
        r = requests.post(upload_url, files={"file": (Path(img_path).name, f, "image/png")})
        file_uris[img_path] = r.json()["file"]["uri"]

# 2. Concurrent Processing with Progress Bar
tasks = []
for img_path, uri in file_uris.items():
    for m in MODELS:
        for p in PROMPTS:
            tasks.append((img_path, uri, m, p))

print(f"Firing {len(tasks)} requests simultaneously...")
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(process_task, *t) for t in tasks]
    # This wraps the futures in a progress bar
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing AI Tasks"):
        result = future.result()
        if "error" not in result:
            all_results.append(result)

# 3. Graphing the Results
if all_results:
    df = pd.DataFrame(all_results)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- Time Graph ---
    avg_times = df.groupby('prompt_id')['time_s'].mean()
    avg_times.plot(kind='bar', ax=ax1, color='#6200ee', edgecolor='black')
    ax1.set_title("Avg Processing Time (s)")
    ax1.set_ylabel("Seconds")
    ax1.set_xticklabels(avg_times.index, rotation=0)
    
    # --- Cost Graph (with $0.000 formatting) ---
    img_costs = df.groupby('image')['cost_usd'].sum()
    total_cost = img_costs.sum()
    
    # Custom function to show absolute $ value and percentage
    def cost_label(pct):
        val = (pct / 100.0) * total_cost
        return f'${val:.3f}\n({pct:.1f}%)'

    img_costs.plot(
        kind='pie', 
        ax=ax2, 
        autopct=cost_label, 
        startangle=140, 
        cmap='Spectral',
        pctdistance=0.75,
        textprops={'fontsize': 10, 'weight': 'bold'}
    )
    
    ax2.set_title(f"Cost per Image (Total: ${total_cost:.3f})")
    ax2.set_ylabel("")

    plt.tight_layout()
    plt.show()

In [ ]:
import json
from pathlib import Path

def consolidate_flat_results(directory_path="data"):
    results_dir = Path(directory_path)
    # Using a list to maintain the distinct record of every prompt run
    master_records = []

    # Find all JSON files
    json_files = sorted(results_dir.glob("*.json"))
    
    for file_path in json_files:
        if file_path.name == "master_results.json":
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Flatten the structure
            record = {
                "image": data.get("image"),
                "model": data.get("model"),
                "prompt_id": data.get("prompt_id"),
                "response": data.get("response"),
                "processing_time_s": data.get("time_s"),
                # Flattening usage metrics
                "prompt_tokens": data.get("usage", {}).get("prompt"),
                "completion_tokens": data.get("usage", {}).get("completion"),
                "cost_usd": data.get("usage", {}).get("cost_usd")
            }
            
            # Attempt to parse response if it's the structured extraction
            if record["prompt_id"] == "structured" and isinstance(record["response"], str):
                try:
                    record["response"] = json.loads(record["response"])
                except:
                    pass
            
            master_records.append(record)
            
        except Exception as e:
            print(f"Error flattening {file_path.name}: {e}")

    # Write the flattened master file
    output_path = results_dir / "master_results.json"
    with open(output_path, "w", encoding='utf-8') as f:
        json.dump(master_records, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Master file created at: {output_path}")
    return master_records

# Run the consolidation
final_data = consolidate_flat_results()

In [ ]:
import json
from pathlib import Path

def generate_improved_dashboard(json_path="data/master_results.json"):
    # Load the consolidated data
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: {json_path} not found. Run the consolidation script first.")
        return

    # Grouping by image for the UI
    grouped = {}
    for entry in data:
        img = entry.get('image', 'Unknown')
        if img not in grouped:
            grouped[img] = []
        grouped[img].append(entry)

    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Technical Analysis | Gemini 2.5 Pro</title>
        <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600&family=Fira+Code:wght@400;500&display=swap" rel="stylesheet">
        <style>
            :root {{
                --bg: #0b0e14;
                --card: #161a23;
                --accent: #6366f1;
                --text: #f8fafc;
                --dim: #94a3b8;
                --border: #2d3748;
            }}
            body {{
                background: var(--bg);
                color: var(--text);
                font-family: 'Inter', sans-serif;
                margin: 0;
                line-height: 1.6;
            }}
            .header {{
                padding: 30px 50px;
                border-bottom: 1px solid var(--border);
                background: rgba(11, 14, 20, 0.8);
                backdrop-filter: blur(10px);
                position: sticky;
                top: 0;
                z-index: 100;
            }}
            .header h1 {{ margin: 0; font-weight: 300; font-size: 1.5rem; }}
            .header b {{ color: var(--accent); }}

            .main-container {{ padding: 20px 50px; }}

            .image-section {{
                display: flex;
                gap: 0;
                margin-bottom: 60px;
                background: var(--card);
                border-radius: 20px;
                border: 1px solid var(--border);
                min-height: 600px;
                overflow: hidden;
            }}
            
            /* 33% Image Column */
            .image-column {{
                flex: 0 0 33%;
                background: #000;
                display: flex;
                flex-direction: column;
                padding: 20px;
                border-right: 1px solid var(--border);
                position: relative;
            }}
            .image-column img {{
                width: 100%;
                height: auto;
                object-fit: contain;
                border-radius: 8px;
                position: sticky;
                top: 100px;
            }}
            .img-label {{
                font-size: 0.7rem;
                color: var(--dim);
                text-align: center;
                margin-top: 15px;
                font-family: 'Fira Code', monospace;
            }}

            /* 67% Content Column */
            .data-column {{
                flex: 1;
                padding: 40px;
                overflow-y: visible;
                word-wrap: break-word; /* Prevents text cutoff */
                overflow-wrap: break-word;
                max-width: 67%;
            }}

            .prompt-block {{
                margin-bottom: 40px;
                animation: fadeIn 0.5s ease forwards;
            }}
            .prompt-header {{
                display: flex;
                justify-content: space-between;
                align-items: center;
                margin-bottom: 15px;
            }}
            .prompt-id {{
                text-transform: uppercase;
                font-size: 0.8rem;
                letter-spacing: 1.5px;
                font-weight: 600;
                color: var(--accent);
            }}
            .meta-group {{ display: flex; gap: 10px; }}
            .pill {{
                background: #1e293b;
                padding: 4px 10px;
                border-radius: 6px;
                font-size: 0.7rem;
                color: var(--dim);
                border: 1px solid #334155;
            }}

            .content-box {{
                background: rgba(15, 17, 23, 0.5);
                padding: 20px;
                border-radius: 12px;
                border: 1px solid #2d3748;
                font-size: 0.95rem;
                color: #cbd5e1;
                white-space: pre-wrap; /* Preserves line breaks in transcription */
            }}
            
            pre {{
                margin: 0;
                font-family: 'Fira Code', monospace;
                font-size: 0.85rem;
                color: #a5b4fc;
                white-space: pre-wrap; /* Forces JSON to wrap */
                word-break: break-all;
            }}

            @keyframes fadeIn {{
                from {{ opacity: 0; transform: translateY(10px); }}
                to {{ opacity: 1; transform: translateY(0); }}
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Vision Intelligence <b>Gemini 2.5 Pro</b></h1>
        </div>
        <div class="main-container">
    """

    for img_name, records in grouped.items():
        html_content += f"""
        <div class="image-section">
            <div class="image-column">
                <img src="{img_name}" alt="Source Asset">
                <div class="img-label">{img_name}</div>
            </div>
            <div class="data-column">
        """
        
        for r in records:
            # Handle potential None/null values gracefully
            prompt_id = r.get('prompt_id', 'N/A')
            model = r.get('model') or 'Unknown'
            time_s = r.get('processing_time_s')
            time_str = f"{time_s:.1f}s" if time_s is not None else "N/A"
            cost = r.get('cost_usd')
            cost_str = f"${cost:.4f}" if cost is not None else "N/A"
            response = r.get('response') or "No response data available."

            # Format response (JSON for 'structured', Text for others)
            display_response = ""
            if prompt_id == "structured":
                if isinstance(response, (dict, list)):
                    display_response = f"<pre>{json.dumps(response, indent=2)}</pre>"
                else:
                    try:
                        parsed = json.loads(response)
                        display_response = f"<pre>{json.dumps(parsed, indent=2)}</pre>"
                    except:
                        display_response = f"<pre>{response}</pre>"
            else:
                display_response = response

            html_content += f"""
                <div class="prompt-block">
                    <div class="prompt-header">
                        <div class="prompt-id">{prompt_id}</div>
                        <div class="meta-group">
                            <span class="pill">{model}</span>
                            <span class="pill">{time_str}</span>
                            <span class="pill">{cost_str}</span>
                        </div>
                    </div>
                    <div class="content-box">{display_response}</div>
                </div>
            """
        
        html_content += "</div></div>"

    html_content += "</div></body></html>"
    
    output_file = Path("data/master_results.html")
    output_file.write_text(html_content, encoding='utf-8')
    print(f"✅ Dashboard generated: {output_file}")

generate_improved_dashboard()

In [ ]:
!curl -L -o "./data/formula-1-world-championship-1950-2020.zip" \
  "https://www.kaggle.com/api/v1/datasets/download/rohanrao/formula-1-world-championship-1950-2020"

In [155]:
import pandas as pd

# Load the files
path = './data/formula-1-world-championship-1950-2020/'
results = pd.read_csv(path + 'results.csv')
drivers = pd.read_csv(path + 'drivers.csv')
constructors = pd.read_csv(path + 'constructors.csv')
races = pd.read_csv(path + 'races.csv')
circuits = pd.read_csv(path + 'circuits.csv')
status = pd.read_csv(path + 'status.csv') # Added to make "status" readable

# 1. Join Drivers to Results
merged_df = pd.merge(results, drivers[['driverId', 'driverRef', 'forename', 'surname']], on='driverId')

# 2. Join Constructors (Teams)
merged_df = pd.merge(merged_df, constructors[['constructorId', 'name', 'nationality']], 
                     on='constructorId', suffixes=('_driver', '_team'))

# 3. Join Race info
merged_df = pd.merge(merged_df, races[['raceId', 'year', 'date', 'time', 'round', 'name', 'circuitId']], 
                     on='raceId', suffixes=('', '_race'))

# 4. Join Circuits (to get the track name)
merged_df = pd.merge(merged_df, circuits[['circuitId', 'name', 'location', 'country']], 
                     on='circuitId', suffixes=('', '_circuit'))

# 5. Join Status (to turn statusId into "Finished", "Engine", etc.)
merged_df = pd.merge(merged_df, status, on='statusId')

In [156]:
merged_df.columns

Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'driverRef', 'forename', 'surname',
       'name', 'nationality', 'year', 'date', 'time_race', 'round',
       'name_race', 'circuitId', 'name_circuit', 'location', 'country',
       'status'],
      dtype='object')

In [157]:
# Rename for clarity before building the sentence
df = merged_df.rename(columns={
    'name': 'team_name',
    'nationality': 'nationality',
    'number': 'car_number',
    'name_circuit': 'circuit_name'
})

In [158]:
df['status'].unique()

array(['Finished', '+1 Lap', 'Engine', 'Collision', 'Accident',
       'Transmission', 'Clutch', 'Electrical', 'Hydraulics',
       'Disqualified', '+2 Laps', 'Spun off', 'Gearbox', 'Radiator',
       'Suspension', '+4 Laps', 'Brakes', '+3 Laps', 'Overheating',
       'Mechanical', 'Tyre', 'Driver Seat', 'Puncture', 'Driveshaft',
       'Retired', 'Fuel pressure', 'Front wing', 'Water pressure',
       'Refuelling', 'Wheel', 'Throttle', 'Steering', 'Technical',
       'Electronics', 'Broken wing', 'Heat shield fire', 'Exhaust',
       'Oil leak', '+11 Laps', 'Wheel rim', 'Water leak', 'Fuel pump',
       'Track rod', '+5 Laps', '+17 Laps', 'Oil pressure', 'Pneumatics',
       'Withdrew', '+12 Laps', '+7 Laps', 'Engine fire', '+26 Laps',
       'Tyre puncture', 'Out of fuel', 'Wheel nut', 'Not classified',
       '+6 Laps', '+8 Laps', 'Handling', 'Rear wing', 'Fire',
       'Fuel system', 'Oil line', 'Fuel rig', 'Launch control', 'Injured',
       'Fuel', 'Power loss', '107% Rule', 'Saf

In [ ]:
import pandas as pd

from datetime import datetime

def format_race_date(date_str):
    """Turn '2009-03-29' into '29 March 2009'."""
    if pd.isna(date_str) or date_str in (None, "", r'\N'):
        return None
    try:
        d = datetime.strptime(str(date_str), "%Y-%m-%d")
        return d.strftime("%-d %B %Y")  # e.g. 29 March 2009 (on Windows use '%#d')
    except ValueError:
        return str(date_str)
    
# ---- Time helpers ---------------------------------------------------------

def verbalize_lap_time(lap_time):
    """Converts '1:29.639' to '1 minute and 29.639 seconds'."""
    if pd.isna(lap_time) or lap_time == r'\N' or ":" not in str(lap_time):
        return None  # signal: no valid lap time
    
    try:
        minutes_str, seconds = str(lap_time).split(":", 1)
        minutes = int(minutes_str)
        minute_word = "minute" if minutes == 1 else "minutes"
        return f"{minutes} {minute_word} and {seconds} seconds"
    except (IndexError, ValueError):
        return None
        

def format_ms_to_readable(ms):
    """Converts milliseconds to a human-readable Hh Mm Ss format."""
    if pd.isna(ms) or ms == "" or ms == r'\N':
        return None  # signal: no race time
    
    ms = int(float(ms))
    seconds = int((ms / 1000) % 60)
    minutes = int((ms / (1000 * 60)) % 60)
    hours = int((ms / (1000 * 60 * 60)) % 24)
    
    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    return f"{minutes}m {seconds}s"


# ---- Status grouping (raw sets) ------------------------------------------

LAP_DOWN_STATUSES = {
    "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps", "+6 Laps",
    "+7 Laps", "+8 Laps", "+9 Laps", "+10 Laps", "+11 Laps", "+12 Laps",
    "+13 Laps", "+14 Laps", "+15 Laps", "+16 Laps", "+17 Laps", "+18 Laps",
    "+19 Laps", "+20 Laps", "+21 Laps", "+22 Laps", "+23 Laps", "+24 Laps",
    "+25 Laps", "+26 Laps", "+29 Laps", "+30 Laps", "+42 Laps", "+44 Laps",
    "+46 Laps",
}

CRASH_STATUSES = {
    "Collision", "Accident", "Spun off", "Collision damage",
    "Debris", "Fatal accident", "Fire",
}

MECHANICAL_STATUSES = {
    "Engine", "Transmission", "Clutch", "Electrical", "Hydraulics",
    "Gearbox", "Radiator", "Suspension", "Brakes", "Overheating",
    "Mechanical", "Tyre", "Driver Seat", "Puncture", "Driveshaft",
    "Fuel pressure", "Front wing", "Water pressure", "Refuelling",
    "Wheel", "Throttle", "Steering", "Technical", "Electronics",
    "Broken wing", "Heat shield fire", "Exhaust", "Oil leak",
    "Wheel rim", "Water leak", "Fuel pump", "Track rod",
    "Oil pressure", "Pneumatics", "Engine fire", "Tyre puncture",
    "Wheel nut", "Rear wing", "Fuel system", "Oil line", "Fuel rig",
    "Launch control", "Drivetrain", "Ignition", "Chassis", "Battery",
    "Halfshaft", "Crankshaft", "Alternator", "Differential",
    "Wheel bearing", "Oil pump", "Fuel leak", "Injection",
    "Distributor", "Turbo", "CV joint", "Water pump", "Spark plugs",
    "Fuel pipe", "Oil pipe", "Axle", "Water pipe", "Magneto",
    "Supercharger", "Engine misfire", "ERS", "Power Unit",
    "Brake duct", "Seat", "Damage", "Cooling system", "Undertray",
}

ADMIN_STATUSES = {
    "Disqualified", "Retired", "Withdrew", "Not classified",
    "107% Rule", "Safety", "Did not qualify", "Did not prequalify",
    "Excluded", "Not restarted", "Underweight", "Safety belt",
}

HEALTH_STATUSES = {
    "Injured", "Injury", "Driver unwell", "Illness", "Physical",
    "Eye injury", "Safety concerns",
}

FUEL_STATUSES = {
    "Out of fuel", "Fuel", "Fuel system", "Fuel pump",
    "Fuel leak", "Fuel pipe", "Fuel rig", "Fuel pressure",
}


# ---- High-level status group (finished vs lapped vs dnf-type) ------------

def classify_status_group(status: str) -> str:
    status = str(status)
    if status == "Finished":
        return "finished"
    if status in LAP_DOWN_STATUSES:
        return "lapped"
    if status in CRASH_STATUSES:
        return "crash"
    if status in FUEL_STATUSES:
        return "fuel"
    if status in MECHANICAL_STATUSES:
        return "mechanical"
    if status in ADMIN_STATUSES:
        return "admin"
    if status in HEALTH_STATUSES:
        return "health"
    return "other"


# ---- Status family / subfamily -------------------------------------------

def classify_status_family(status: str) -> str:
    """Coarse families you can filter on: engine, powertrain, brakes, crash, fuel, admin, health, etc."""
    status = str(status)

    if status in CRASH_STATUSES:
        return "crash"
    if status in FUEL_STATUSES:
        return "fuel"
    if status in ADMIN_STATUSES:
        return "admin"
    if status in HEALTH_STATUSES:
        return "health"
    if status in LAP_DOWN_STATUSES:
        return "lapped"
    if status == "Finished":
        return "finished"

    if status in MECHANICAL_STATUSES:
        # mechanical families
        if status in {"Transmission", "Gearbox", "Clutch", "Drivetrain", "Differential",
                      "Halfshaft", "Driveshaft", "CV joint"}:
            return "powertrain"
        if status in {"Engine", "Engine misfire", "Turbo", "Supercharger", "Power Unit"}:
            return "engine"
        if status in {"ERS"}:
            return "ers"
        if status in {"Brakes", "Brake duct"}:
            return "brakes"
        if status in {"Tyre", "Tyre puncture", "Puncture",
                      "Wheel", "Wheel nut", "Wheel rim", "Wheel bearing"}:
            return "wheels_tyres"
        if status in {"Suspension", "Chassis", "Undertray"}:
            return "suspension_chassis"
        if status in {"Radiator", "Cooling system"}:
            return "cooling"
        if status in {"Electrical", "Electronics", "Battery", "Ignition", "Magneto"}:
            return "electrical"
        if status in {"Oil leak", "Oil pressure", "Oil line", "Oil pump", "Oil pipe"}:
            return "oil_system"
        if status in {"Water leak", "Water pressure", "Water pump", "Water pipe"}:
            return "water_system"
        if status in {"Front wing", "Rear wing", "Broken wing"}:
            return "aero_bodywork"
        # everything else mechanical
        return "mechanical_other"

    return "other"


def classify_status_subfamily(status: str) -> str:
    """More specific sub-families for fine-grained querying, e.g. 'gearbox', 'clutch', 'driveshaft'."""
    status = str(status)

    # Powertrain subfamilies
    if status in {"Transmission"}:
        return "transmission"
    if status in {"Gearbox"}:
        return "gearbox"
    if status in {"Clutch"}:
        return "clutch"
    if status in {"Drivetrain"}:
        return "drivetrain"
    if status in {"Differential"}:
        return "differential"
    if status in {"Halfshaft"}:
        return "halfshaft"
    if status in {"Driveshaft"}:
        return "driveshaft"
    if status in {"CV joint"}:
        return "cv_joint"

    # Engine & ERS subfamilies
    if status in {"Engine", "Engine misfire"}:
        return "engine_general"
    if status in {"Turbo"}:
        return "turbo"
    if status in {"Supercharger"}:
        return "supercharger"
    if status in {"Power Unit"}:
        return "power_unit"
    if status in {"ERS"}:
        return "ers"

    # Brakes
    if status in {"Brakes"}:
        return "brakes"
    if status in {"Brake duct"}:
        return "brake_duct"

    # Wheels / tyres
    if status in {"Tyre", "Tyre puncture", "Puncture"}:
        return "tyre"
    if status in {"Wheel"}:
        return "wheel"
    if status in {"Wheel nut"}:
        return "wheel_nut"
    if status in {"Wheel rim"}:
        return "wheel_rim"
    if status in {"Wheel bearing"}:
        return "wheel_bearing"

    # Fluids / cooling
    if status in {"Oil leak", "Oil pressure", "Oil line", "Oil pump", "Oil pipe"}:
        return "oil_system"
    if status in {"Water leak", "Water pressure", "Water pump", "Water pipe"}:
        return "water_system"
    if status in {"Radiator", "Cooling system"}:
        return "cooling"

    # Electrical
    if status in {"Electrical", "Electronics", "Battery", "Ignition", "Magneto"}:
        return "electrical"

    # Suspension / chassis
    if status in {"Suspension"}:
        return "suspension"
    if status in {"Chassis"}:
        return "chassis"
    if status in {"Undertray"}:
        return "undertray"

    # Aero / body
    if status in {"Front wing"}:
        return "front_wing"
    if status in {"Rear wing"}:
        return "rear_wing"
    if status in {"Broken wing"}:
        return "broken_wing"

    # Crash, fuel, health, admin could be used as-is
    if status in CRASH_STATUSES:
        return "crash"
    if status in FUEL_STATUSES:
        return "fuel"
    if status in HEALTH_STATUSES:
        return "health"
    if status in ADMIN_STATUSES:
        return "admin"
    if status in LAP_DOWN_STATUSES:
        return "lapped"
    if status == "Finished":
        return "finished"

    return "other"


# ---- Outcome template logic ----------------------------------------------

def generate_outcome_segment(row):
    full_name = f"{row['forename']} {row['surname']}"
    status = str(row['status'])
    pos_text = str(row['positionText'])
    points = int(float(row['points']))

    group = classify_status_group(status)

    # Numeric positionText → classified finisher (including lapped)
    if pos_text.isdigit():
        p = int(pos_text)
        podium = " on the podium" if p <= 3 else ""
        if group == "finished":
            return (
                f"{full_name} finished the race{podium} in position P{p}, "
                f"scoring {points} championship points."
            )
        if group == "lapped":
            return (
                f"{full_name} was classified in position P{p}{podium}, "
                f"one or more laps down on the winner, and scored "
                f"{points} points."
            )
        # Classified but with a non‑'Finished' status (data oddity)
        return (
            f"{full_name} was classified in position P{p}{podium} with "
            f"a status of '{status}', earning {points} points."
        )

    # Non‑numeric positionText → DNF / NC templates by status group
    if group == "crash":
        return (
            f"{full_name} retired from the race after an incident "
            f"({status}), and was not classified as a finisher, "
            f"earning {points} points."
        )
    if group == "mechanical":
        return (
            f"{full_name} retired due to a {status.lower()} issue, "
            f"and was not classified at the finish, scoring "
            f"{points} points."
        )
    if group == "fuel":
        return (
            f"{full_name} was forced to retire with a fuel‑related "
            f"problem ('{status}'), and did not reach the finish, "
            f"scoring {points} points."
        )
    if group == "health":
        return (
            f"{full_name} did not complete the race due to "
            f"{status.lower()}, and was not classified, earning "
            f"{points} points."
        )
    if group == "admin":
        return (
            f"{full_name} was removed from the final classification "
            f"('{status}'), and therefore did not finish the race, "
            f"scoring {points} points."
        )

    # Fallback for odd codes
    return (
        f"{full_name} did not finish the race and was listed with "
        f"a status of '{status}', earning {points} points."
    )


# ---- Main narrative generator --------------------------------------------

def generate_f1_narrative(row):
    full_name = f"{row['forename']} {row['surname']}"
    race_date_raw = row.get('date')
    race_time = row.get('time')

    pretty_date = format_race_date(race_date_raw)

    if pretty_date:
        if pd.notna(race_time) and race_time not in (r'\N', "", None):
            date_clause = f" on {pretty_date}"
        else:
            date_clause = f" on {pretty_date}"
    else:
        date_clause = ""

    # Race time
    readable_total_time = format_ms_to_readable(row['milliseconds'])
    raw_time_gap = str(row['time_y']) if 'time_y' in row else str(row['time'])
    raw_time_gap = None if raw_time_gap == r'\N' else raw_time_gap

    if readable_total_time:
        if raw_time_gap:
            time_segment = (
                f" and completed {row['laps']} laps in a total time of "
                f"{readable_total_time}."
            )
        else:
            time_segment = (
                f" and completed {row['laps']} laps in a total time of "
                f"{readable_total_time}."
            )
    else:
        if raw_time_gap:
            time_segment = (
                f" and was classified after completing {row['laps']} laps, "
                f"finishing {raw_time_gap} behind the leader."
            )
        else:
            time_segment = (
                f" and was classified after completing {row['laps']} laps, "
                f"with no official race time recorded."
            )

    # Fastest lap
    f_lap_time_verbal = verbalize_lap_time(row['fastestLapTime'])
    f_lap_speed = (
        row['fastestLapSpeed'] if str(row['fastestLapSpeed']) != r'\N' else None
    )

    if f_lap_time_verbal and f_lap_speed:
        fastest_lap_segment = (
            f"The fastest lap for {full_name} was lap {row['fastestLap']} "
            f"at {f_lap_time_verbal}, with an average speed of "
            f"{f_lap_speed} km/h."
        )
    elif f_lap_time_verbal:
        fastest_lap_segment = (
            f"The fastest lap for {full_name} was lap {row['fastestLap']} "
            f"at {f_lap_time_verbal}."
        )
    elif f_lap_speed:
        fastest_lap_segment = (
            f"{full_name}'s recorded fastest lap had an average speed of "
            f"{f_lap_speed} km/h."
        )
    else:
        fastest_lap_segment = (
            f"No valid fastest lap time was recorded for {full_name}."
        )

    outcome = generate_outcome_segment(row)

    sentence = (
        f"In round {row['round']} of the {row['year']} Formula 1 World Championship, "
        f"the {row['name_race']} was held at {row['circuit_name']} in "
        f"{row['location']}{date_clause}. "
        f"{full_name} drove for the {row['team_name']} team in car number {row['car_number']}. "
        f"{full_name} started from grid position P{row['grid']}{time_segment} "
        f"{outcome} {fastest_lap_segment}"
    )
    return sentence




# ---- Apply to your DataFrame & add structured fields ---------------------

# Narrative text
df['text_for_opensearch'] = df.apply(generate_f1_narrative, axis=1)


# Structured status fields for querying
df['status_group'] = df['status'].apply(classify_status_group)       # finished / lapped / crash / mechanical / fuel / admin / health / other
df['status_family'] = df['status'].apply(classify_status_family)     # engine / powertrain / brakes / crash / fuel / etc.
df['status_subfamily'] = df['status'].apply(classify_status_subfamily)  # gearbox / transmission / clutch / driveshaft / ers / turbo / etc.
df['finished_flag'] = df['status_group'].isin(['finished', 'lapped'])
df['dnf_flag'] = ~df['finished_flag']


In [163]:
df.columns


Index(['resultId', 'raceId', 'driverId', 'constructorId', 'car_number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'driverRef', 'forename', 'surname',
       'team_name', 'nationality', 'year', 'date', 'time_race', 'round',
       'name_race', 'circuitId', 'circuit_name', 'location', 'country',
       'status', 'text_for_opensearch', 'status_group', 'status_family',
       'status_subfamily', 'finished_flag', 'dnf_flag'],
      dtype='object')

In [168]:
df['text_for_opensearch'][0]


'In round 1 of the 2008 Formula 1 World Championship, the Australian Grand Prix was held at Albert Park Grand Prix Circuit in Melbourne on 16 March 2008. Lewis Hamilton drove for the McLaren team in car number 22. Lewis Hamilton started from grid position P1 and completed 58 laps in a total time of 1h 34m 50s. Lewis Hamilton finished the race on the podium in position P1, scoring 10 championship points. The fastest lap for Lewis Hamilton was lap 39 at 1 minute and 27.452 seconds, with an average speed of 218.300 km/h.'

In [169]:
df[:10].to_json('example.json', orient='records')


In [170]:
df.to_json('data/formula-1-world-championship-1950-2020.json', orient='records')

## indexing

In [ ]:
from google import genai
from google.genai import types

client_genai = genai.Client(api_key='REDACTED_SEE_ENV')

In [182]:
import os
import time
import json
import pandas as pd
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types


# -------------------------------------------------------------------
# Config
# -------------------------------------------------------------------

OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OPENSEARCH_USER = os.getenv("OPENSEARCH_USER", "admin")
OPENSEARCH_PASS = os.getenv("OPENSEARCH_PASS", "admin")
INDEX_NAME = os.getenv("OPENSEARCH_INDEX", "hybrid-search-index")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "REDACTED_SEE_ENV")
JSON_PATH = os.getenv("F1_JSON_PATH", "example.json")

client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "REDACTED_SEE_ENV")

client_genai = genai.Client(api_key=GEMINI_API_KEY)

model_384 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model_1024 = SentenceTransformer("BAAI/bge-large-en-v1.5")


# -------------------------------------------------------------------
# Embeddings
# -------------------------------------------------------------------

def get_cloud_pair(text: str):
    model_name = "models/gemini-embedding-001"
    try:
        res1 = client_genai.models.embed_content(
            model=model_name,
            contents=text,
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT",
                output_dimensionality=1536,
            ),
        )
        res2 = client_genai.models.embed_content(
            model=model_name,
            contents=text,
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT",
                output_dimensionality=3072,
            ),
        )
        v1536 = res1.embeddings[0].values
        v3072 = res2.embeddings[0].values
        return v1536, v3072
    except Exception as e:
        print(f"Cloud API Error: {e}")
        return [0.0] * 1536, [0.0] * 3072


# -------------------------------------------------------------------
# Index from prebuilt JSON
# -------------------------------------------------------------------

def load_json_as_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data)


def index_prebuilt_json(json_path: str, batch_size: int = 50, max_workers: int = 10):
    df = load_json_as_df(json_path)

    if "text_for_opensearch" not in df.columns:
        raise ValueError("JSON must contain 'text_for_opensearch' field")

    total = len(df)
    print(f"Indexing {total} rows from {json_path} with {max_workers} threads...")

    texts = df["text_for_opensearch"].tolist()

    for i in range(0, total, batch_size):
        batch_df = df.iloc[i: i + batch_size].copy()
        batch_texts = batch_df["text_for_opensearch"].tolist()

        # Local embeddings
        v384_list = model_384.encode(batch_texts).tolist()
        v1024_list = model_1024.encode(batch_texts).tolist()

        # Cloud embeddings
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            cloud_results = list(executor.map(get_cloud_pair, batch_texts))

        actions = []
        for idx, (index_val, row) in enumerate(batch_df.iterrows()):
            v1536, v3072 = cloud_results[idx]

            # Entities (lightweight extraction from existing fields)
            entity_list = [
                f"{row.get('forename','')} {row.get('surname','')}",
                row.get("driverRef"),
                row.get("team_name"),
                row.get("nationality"),
                row.get("name_race"),
                row.get("status"),
                row.get("location"),
                row.get("circuit_name"),
                str(row.get("year")),
            ]
            clean_entities = ", ".join(
                [str(e) for e in entity_list if e not in (None, "", r"\N", "\\N")]
            )

            metadata = {
                col: (None if str(row[col]) in (r"\N", "\\N") else row[col])
                for col in batch_df.columns
            }

            doc = {
                "_index": INDEX_NAME,
                "_id": f"f1_{row['resultId']}",
                "_op_type": "index",
                "doc_type": "f1_result",
                "security_label": "PUBLIC",
                "security_label_2": "UNRESTRICTED",
                "timestamp": f"{int(row['year'])}-01-01T00:00:00Z",
                "text_representation": row["text_for_opensearch"],
                "entities": clean_entities,
                "embedding_384": v384_list[idx],
                "embedding_1024": v1024_list[idx],
                "embedding_1536": v1536,
                "embedding_3072": v3072,
                "metadata": metadata,
            }
            actions.append(doc)

        success, failed = helpers.bulk(client, actions, raise_on_error=False)
        print(f"Batch {i}-{i + len(batch_df)}: success={success}, failed={failed}")
        time.sleep(0.5)


if __name__ == "__main__":
    index_prebuilt_json(JSON_PATH)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexing 10 rows from example.json with 10 threads...
Batch 0-10: success=10, failed=[]


### confirm stuff was indexed

In [187]:
from opensearchpy import OpenSearch

client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False,          
    verify_certs=False,
    ssl_show_warn=False,
)

index_name = "hybrid-search-index"

# Simple "match_all" to see first 5 docs
resp = client.search(
    index=index_name,
    body={
        "size": 5,
        "query": {"match_all": {}}
    },
)

for hit in resp["hits"]["hits"]:
    print(hit["_id"], hit["_source"]["text_representation"])


f1_1 In round 1 of the 2008 Formula 1 World Championship, the Australian Grand Prix was held at Albert Park Grand Prix Circuit in Melbourne on 16 March 2008. Lewis Hamilton drove for the McLaren team in car number 22. Lewis Hamilton started from grid position P1 and completed 58 laps in a total time of 1h 34m 50s. Lewis Hamilton finished the race on the podium in position P1, scoring 10 championship points. The fastest lap for Lewis Hamilton was lap 39 at 1 minute and 27.452 seconds, with an average speed of 218.300 km/h.
f1_2 In round 1 of the 2008 Formula 1 World Championship, the Australian Grand Prix was held at Albert Park Grand Prix Circuit in Melbourne on 16 March 2008. Nick Heidfeld drove for the BMW Sauber team in car number 3. Nick Heidfeld started from grid position P5 and completed 58 laps in a total time of 1h 34m 56s. Nick Heidfeld finished the race on the podium in position P2, scoring 8 championship points. The fastest lap for Nick Heidfeld was lap 41 at 1 minute and 27

### test indexing querying

In [188]:
import os
from typing import List, Tuple

from google import genai
from google.genai import types
from opensearchpy import OpenSearch

# -------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "REDACTED_SEE_ENV")

GEMINI_EMBED_MODEL = "gemini-embedding-001"          # embeddings[web:45]
GEMINI_RAG_MODEL   = "gemini-2.5-flash"             # text generation[web:34][web:36]

OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_USER = "admin"
OPENSEARCH_PASS = "admin"
OPENSEARCH_INDEX = "hybrid-search-index"            # must match your index
EMBEDDING_FIELD = "embedding_1024"                  # choose the field you want

TOP_K = 5

# -------------------------------------------------------------------
# 2. Clients
# -------------------------------------------------------------------

client_genai = genai.Client(api_key=GEMINI_API_KEY)

client_os = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
)

# -------------------------------------------------------------------
# 3. Embedding + vector search
# -------------------------------------------------------------------

def embed_query(text: str) -> List[float]:
    """Create a query embedding with Gemini."""
    res = client_genai.models.embed_content(
        model=GEMINI_EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=1024,
        ),
    )
    return res.embeddings[0].values


def semantic_search(query_text: str, top_k: int = TOP_K) -> Tuple[List[str], list]:
    """Pure kNN search on one embedding field, no hybrid."""
    vec = embed_query(query_text)

    search_body = {
        "size": top_k,
        "query": {
            "knn": {
                EMBEDDING_FIELD: {
                    "vector": vec,
                    "k": top_k,
                }
            }
        },
        "_source": ["text_representation", "metadata"],
    }

    res = client_os.search(index=OPENSEARCH_INDEX, body=search_body)
    hits = res["hits"]["hits"]

    contexts = [h["_source"]["text_representation"] for h in hits]
    return contexts, hits

# -------------------------------------------------------------------
# 4. RAG answer with gemini-2.5-flash
# -------------------------------------------------------------------

def build_rag_prompt(question: str, contexts: List[str]) -> str:
    context_block = "\n\n".join(
        f"Document {i+1}:\n{c}" for i, c in enumerate(contexts)
    )
    prompt = (
        "You are an F1 racing expert. Use ONLY the context documents to answer.\n\n"
        f"{context_block}\n\n"
        f"Question: {question}\n\n"
        "If the answer cannot be found in the documents, say you don't know."
    )
    return prompt


def answer_with_rag(question: str) -> str:
    contexts, hits = semantic_search(question)
    if not contexts:
        return "No results found in the vector index."

    prompt = build_rag_prompt(question, contexts)

    resp = client_genai.models.generate_content(
        model=GEMINI_RAG_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
        ),
    )

    return resp.text

# -------------------------------------------------------------------
# 5. Example usage
# -------------------------------------------------------------------

if __name__ == "__main__":
    q1 = "what happened at Australian Grand Prix?"
    print("Q:", q1)
    print("A:", answer_with_rag(q1))

    print("\n---\n")

    q2 = "who won Australian Grand Prix?"
    print("Q:", q2)
    print("A:", answer_with_rag(q2))


Q: what happened at Australian Grand Prix?


ConnectTimeout: [Errno 60] Operation timed out

In [189]:
def debug_retrieval(question: str, top_k: int = 5):
    contexts, hits = semantic_search(question)  
    print(f"\nQUESTION: {question}")
    print(f"hits: {len(hits)}")
    for i, h in enumerate(hits):
        print(f"\n--- HIT {i+1} (score={h['_score']:.4f}) ---")
        print(h["_source"]["text_representation"])
        
debug_retrieval("who won Australian Grand Prix?")
debug_retrieval("tell me what happened at the Australian Grand Prix in detail")



QUESTION: who won Australian Grand Prix?
hits: 5

--- HIT 1 (score=0.4236) ---
In round 1 of the 2008 Formula 1 World Championship, the Australian Grand Prix was held at Albert Park Grand Prix Circuit in Melbourne on 16 March 2008. Robert Kubica drove for the BMW Sauber team in car number 4. Robert Kubica started from grid position P2 and was classified after completing 47 laps, with no official race time recorded. Robert Kubica retired from the race after an incident (Collision), and was not classified as a finisher, earning 0 points. The fastest lap for Robert Kubica was lap 15 at 1 minute and 28.753 seconds, with an average speed of 215.100 km/h.

--- HIT 2 (score=0.4226) ---
In round 1 of the 2008 Formula 1 World Championship, the Australian Grand Prix was held at Albert Park Grand Prix Circuit in Melbourne on 16 March 2008. Nico Rosberg drove for the Williams team in car number 7. Nico Rosberg started from grid position P7 and completed 58 laps in a total time of 1h 34m 58s. Nico

## Verbalizing the data

To minimize the signal for the race in the verbalization when per driver per race, verbalizing aggregated race

In [208]:
api_key = 'REDACTED_SEE_ENV'

In [210]:
import os
import time
import pandas as pd
from datetime import datetime
from google import genai
from google.genai import types
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer

# 1. Setup
client_genai = genai.Client(api_key=api_key)
client_os = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False, verify_certs=False, ssl_show_warn=False
)

model_384 = SentenceTransformer('all-MiniLM-L6-v2')
model_1024 = SentenceTransformer('BAAI/bge-large-en-v1.5')

# 2. Re-incorporate your status classification logic
def classify_status_group(status: str) -> str:
    status = str(status)
    if status == "Finished": return "finished"
    if status in LAP_DOWN_STATUSES: return "lapped"
    if status in CRASH_STATUSES: return "crash"
    if status in FUEL_STATUSES: return "fuel"
    if status in MECHANICAL_STATUSES: return "mechanical"
    if status in ADMIN_STATUSES: return "admin"
    if status in HEALTH_STATUSES: return "health"
    return "other"

def format_race_date(date_str):
    if pd.isna(date_str) or date_str in (None, "", r'\N'): return None
    try:
        d = datetime.strptime(str(date_str), "%Y-%m-%d")
        return d.strftime("%-d %B %Y")
    except ValueError: return str(date_str)

# 3. Apply Aggregate Intelligence to create df_races
def generate_race_narrative_with_intelligence(group):
    row = group.iloc[0]
    
    # Retirement/DNF Flags
    group['status_group'] = group['status'].apply(classify_status_group)
    group['dnf_flag'] = ~group['status_group'].isin(['finished', 'lapped'])
    
    # Podium
    podium = group[group['positionOrder'].isin([1, 2, 3])].sort_values('positionOrder')
    podium_text = ", ".join([f"{p['surname']} (P{int(p['positionOrder'])})" for _, p in podium.iterrows()])
    
    # DNFs
    dnfs = group[group['dnf_flag'] == True]
    mech_dnfs = dnfs[dnfs['status_group'] == 'mechanical']
    crash_dnfs = dnfs[dnfs['status_group'] == 'crash']
    
    retirement_summary = (
        f"There were {len(dnfs)} total retirements. "
        f"Mechanical issues claimed {len(mech_dnfs)} cars "
        f"({', '.join(mech_dnfs['status'].unique()) if not mech_dnfs.empty else 'none'}). "
        f"Crashes and collisions accounted for {len(crash_dnfs)} retirements."
    )

    # Grid
    classification = []
    for _, r in group.sort_values('positionOrder').iterrows():
        status_note = f" [DNF: {r['status']}]" if r['dnf_flag'] else ""
        classification.append(f"P{r['positionOrder']}: {r['surname']}{status_note}")
    
    grid_str = " | ".join(classification)
    pretty_date = format_race_date(row['date'])

    narrative = (
        f"The {row['year']} {row['name_race']} was held on {pretty_date} at {row['circuit_name']} in {row['location']}. "
        f"Winner: {podium.iloc[0]['surname'] if not podium.empty else 'N/A'}. "
        f"Podium: {podium_text}. {retirement_summary} "
        f"Full Results: {grid_str}."
    )
    
    return pd.Series({
        'raceId': row['raceId'],
        'year': row['year'], # Fixed: Included to prevent KeyError
        'text_representation': narrative,
        'entities': f"{row['name_race']} {row['year']}, Winner: {podium.iloc[0]['surname'] if not podium.empty else 'N/A'}, DNFs: {', '.join(dnfs['status'].unique())}",
        'winner': podium.iloc[0]['surname'] if not podium.empty else "N/A"
    })

# Pivot the data
df_races = df.groupby('raceId').apply(generate_race_narrative_with_intelligence).reset_index(drop=True)

# 4. Corrected Indexing Function
def index_summaries_to_opensearch(df, batch_size=50):
    print(f"Indexing {len(df)} race summaries...")
    
    for i in range(0, len(df), batch_size):
        batch_df = df.iloc[i:i+batch_size]
        texts = batch_df['text_representation'].tolist()
        
        # Vectors
        v384 = model_384.encode(texts).tolist()
        v1024 = model_1024.encode(texts).tolist()
        
        # Cloud (Batching)
        model_id = "gemini-embedding-001"
        res_1536 = client_genai.models.embed_content(
            model=model_id, contents=texts,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=1536)
        )
        res_3072 = client_genai.models.embed_content(
            model=model_id, contents=texts,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=3072)
        )

        actions = []
        for idx, (_, row) in enumerate(batch_df.iterrows()):
            actions.append({
                "_index": "hybrid-search-index",
                "_id": f"race_{row['raceId']}", 
                "doc_type": "race_summary",
                "security_label": "PUBLIC",
                "timestamp": f"{int(row['year'])}-01-01T00:00:00Z", # Now works
                "text_representation": row['text_representation'],
                "entities": row['entities'],
                "embedding_384": v384[idx],
                "embedding_1024": v1024[idx],
                "embedding_1536": res_1536.embeddings[idx].values,
                "embedding_3072": res_3072.embeddings[idx].values,
                "metadata": {"raceId": int(row['raceId']), "year": int(row['year']), "winner": row['winner']}
            })
            
        helpers.bulk(client_os, actions)
        print(f"Progress: {i + len(batch_df)} / {len(df)}")

# Execute
index_summaries_to_opensearch(df_races)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/24/y__mx0xd3rn5g48sf3wlvf080000gn/T/ipykernel_19047/2494954290.py:89: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_races = df.groupby('raceId').apply(generate_race_narrative_with_intelligence).reset_index(drop=True)


Indexing 1125 race summaries...
Progress: 50 / 1125
Progress: 100 / 1125
Progress: 150 / 1125
Progress: 200 / 1125
Progress: 250 / 1125
Progress: 300 / 1125
Progress: 350 / 1125
Progress: 400 / 1125
Progress: 450 / 1125
Progress: 500 / 1125
Progress: 550 / 1125
Progress: 600 / 1125
Progress: 650 / 1125
Progress: 700 / 1125
Progress: 750 / 1125
Progress: 800 / 1125
Progress: 850 / 1125
Progress: 900 / 1125
Progress: 950 / 1125
Progress: 1000 / 1125
Progress: 1050 / 1125
Progress: 1100 / 1125
Progress: 1125 / 1125


In [212]:
# Confirm indexing

from opensearchpy import OpenSearch

client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False,          
    verify_certs=False,
    ssl_show_warn=False,
)

index_name = "hybrid-search-index"

# Simple "match_all" to see first 5 docs
resp = client.search(
    index=index_name,
    body={
        "size": 5,
        "query": {"match_all": {}}
    },
)

for hit in resp["hits"]["hits"]:
    print(hit["_id"], hit["_source"]["text_representation"])


race_1 The 2009 Australian Grand Prix was held on 29 March 2009 at Albert Park Grand Prix Circuit in Melbourne. Winner: Button. Podium: Button (P1), Barrichello (P2), Trulli (P3). There were 8 total retirements. Mechanical issues claimed 2 cars (Differential, Suspension). Crashes and collisions accounted for 5 retirements. Full Results: P1: Button | P2: Barrichello | P3: Trulli | P4: Glock | P5: Alonso | P6: Rosberg | P7: Buemi | P8: Bourdais | P9: Sutil | P10: Heidfeld | P11: Fisichella | P12: Webber | P13: Vettel [DNF: Collision] | P14: Kubica [DNF: Collision] | P15: Räikkönen [DNF: Differential] | P16: Massa [DNF: Suspension] | P17: Piquet Jr. [DNF: Spun off] | P18: Nakajima [DNF: Accident] | P19: Kovalainen [DNF: Collision] | P20: Hamilton [DNF: Disqualified].
race_2 The 2009 Malaysian Grand Prix was held on 5 April 2009 at Sepang International Circuit in Kuala Lumpur. Winner: Button. Podium: Button (P1), Heidfeld (P2), Glock (P3). There were 5 total retirements. Mechanical issues 

In [223]:
def ask_f1_greedy_chat(question):
    print(f"Question: {question}\n" + "-"*50)
    
    # 1. Fetch the Top 40 results (enough for any driver's career wins)
    TOP_K = 40 
    
    query_res = client_genai.models.embed_content(
        model="gemini-embedding-001",
        contents=question,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=3072)
    )

    search_body = {
        "size": TOP_K,
        "query": {
            "hybrid": {
                "queries": [
                    { "multi_match": { "query": question, "fields": ["text_representation", "entities^5"] } },
                    { "knn": { "embedding_3072": { "vector": query_res.embeddings[0].values, "k": TOP_K } } }
                ]
            }
        },
        "post_filter": { "term": { "doc_type": "race_summary" } }
    }

    response = client_os.search(index="hybrid-search-index", body=search_body)
    
    # 2. Compile these 40 summaries into a temporary "History Slice"
    context_list = []
    for hit in response['hits']['hits']:
        # Help the LLM identify distinct races
        meta = hit['_source'].get('metadata', {})
        header = f"--- {meta.get('year')} {hit['_source'].get('entities','').split(',')[0]} ---"
        context_list.append(f"{header}\n{hit['_source']['text_representation']}")
    
    full_context = "\n\n".join(context_list)

    # 3. Let the LLM do the counting
    prompt = f"""
    You are an F1 Historian. Use the provided race summaries to answer the question.
    If the user asks for a count (e.g., 'how many'), scan the summaries and count the occurrences.
    
    CONTEXT:
    {full_context}

    QUESTION:
    {question}
    """

    answer_res = client_genai.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )

    print(f"ANSWER: {answer_res.text}")
    print("-" * 50)

# NOW TEST THE CAREER COUNT
ask_f1_greedy_chat("how many times did Massa finish P1 in his career?")

Question: how many times did Massa finish P1 in his career?
--------------------------------------------------
ANSWER: Massa finished P1 11 times in the provided race summaries.

--------------------------------------------------


In [220]:
from google.genai import types

client_genai = genai.Client(api_key=api_key)

def ask_f1_chat(question):
    print(f"Question: {question}\n" + "-"*50)
    
    # 1. Generate Query Embedding
    model_id = "gemini-embedding-001"
    query_res = client_genai.models.embed_content(
        model=model_id,
        contents=question,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=3072)
    )
    query_vector = query_res.embeddings[0].values

    # 2. Hybrid Search
    search_body = {
        "size": 200,
        "query": {
            "hybrid": {
                "queries": [
                    {
                        "multi_match": {
                            "query": question,
                            "fields": ["text_representation", "entities^5"]
                        }
                    },
                    {
                        "knn": {
                            "embedding_3072": {
                                "vector": query_vector,
                                "k": 5
                            }
                        }
                    }
                ]
            }
        },
        "post_filter": {
            "term": { "doc_type": "race_summary" }
        }
    }

    response = client_os.search(
        index="hybrid-search-index",
        params={"search_pipeline": "hybrid-search-pipeline"},
        body=search_body
    )

    if not response['hits']['hits']:
        print("No relevant race data found.")
        return

    best_hit = response['hits']['hits'][0]['_source']
    context = best_hit['text_representation']
    
    # --- FIXED ACCESS LOGIC ---
    # Access nested metadata safely
    metadata = best_hit.get('metadata', {})
    year = metadata.get('year', 'Unknown Year')
    
    # Try to extract the race name from the entities string if missing from metadata
    race_info = best_hit.get('entities', '').split(',')[0]
    # ---------------------------

    prompt = f"""
    You are an F1 History Expert. Use the provided race summary to answer the question accurately. If the inquiry is about a specific race, do not try and find a similar one, and instead say "no race that year". Similar for drivers and other queries.
    CONTEXT:
    {context}

    QUESTION:
    {question}
    """

    answer_res = client_genai.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )

    print(f"REVEALED CONTEXT: {year} {race_info}")
    print(f"ANSWER: {answer_res.text}")
    print("-" * 50)

# --- RUN THE TESTS ---

# Test 1: Specific position fact
ask_f1_chat("who was 4th at the 1977 grand prix in australia")

# Test 2: Team performance summary
ask_f1_chat("how did mclaren do in 1993")

# Test 3: Technical failure/DNF query
ask_f1_chat("what happened to ferrari in the 2008 australian grand prix?")

Question: who was 4th at the 1977 grand prix in australia
--------------------------------------------------
REVEALED CONTEXT: 1977 South African Grand Prix 1977
ANSWER: no race that year

--------------------------------------------------
Question: how did mclaren do in 1993
--------------------------------------------------
REVEALED CONTEXT: 1993 Monaco Grand Prix 1993
ANSWER: At the 1993 Monaco Grand Prix, McLaren driver Michael Andretti finished in 8th place, and their other driver, Mark Blundell, retired due to spinning off.

--------------------------------------------------
Question: what happened to ferrari in the 2008 australian grand prix?
--------------------------------------------------
REVEALED CONTEXT: 2008 Australian Grand Prix 2008
ANSWER: Both Ferrari drivers failed to finish the 2008 Australian Grand Prix. Massa retired with an engine failure, and Räikkönen also retired with an engine failure.

--------------------------------------------------


In [221]:
ask_f1_chat("how many times did Massa finish on the podium")

Question: how many times did Massa finish on the podium
--------------------------------------------------
REVEALED CONTEXT: 2008 Brazilian Grand Prix 2008
ANSWER: In the 2008 Brazilian Grand Prix, Massa finished on the podium 1 time.

--------------------------------------------------


In [229]:
ask_f1_greedy_chat("what teams have won the Australian Grand Prix?")

Question: what teams have won the Australian Grand Prix?
--------------------------------------------------
ANSWER: Based on the provided race summaries, the following teams have won the Australian Grand Prix:

*   **Ferrari:** (Schumacher 2000, Schumacher 2004, Vettel 2018, Vettel 2017)
*   **Williams:** (Hill 1995, Hill 1996, Rosberg 1985, Boutsen 1989, Mansell 1994)
*   **McLaren:** (Senna 1991, Häkkinen 1998, Coulthard 2003, Button 2012, Button 2010, Häkkinen 1998, Senna 1993, Hamilton 2008, Hamilton 2015)
*   **Benetton:** (Berger 1992, Schumacher 1996, Piquet 1990)
*   **Ligier:** (Prost 1986)
*   **Jordan:** (Irvine 1999)
*   **Brawn GP:** (Button 2009)
*   **Mercedes:** (Rosberg 2016, Rosberg 2014, Hamilton 2015)
*   **Sauber/Alfa Romeo:** (Räikkönen 2013)
*   **Red Bull Racing:** (Vettel 2011, Verstappen 2023)
*   **Ferrari:** (Sainz 2024)

--------------------------------------------------


In [230]:
ask_f1_greedy_chat("how many times has mclaren won an F1 race?")

Question: how many times has mclaren won an F1 race?
--------------------------------------------------
ANSWER: McLaren has won an F1 race 6 times in the provided summaries.

--------------------------------------------------


In [231]:
# 1. Compile all race summaries into one giant variable
# This creates your "History of F1" document in memory
f1_full_history = "\n\n".join(df_races['text_representation'].tolist())

def ask_f1_total_history(question):
    print(f"Question: {question}\n" + "-"*50)
    
    # We skip OpenSearch entirely and send the full history to Gemini
    # Gemini 2.0 Flash can handle this massive 'needle in a haystack' task easily
    prompt = f"""
    You are an F1 History Expert. Below is the complete historical record of every F1 race.
    Use this data to answer the user's question. 
    
    If the user asks for a count (e.g., 'how many times'), scan the entire history 
    and provide the exact number and a few notable examples.
    
    HISTORY DATA:
    {f1_full_history}

    QUESTION:
    {question}
    """

    # Use gemini-2.0-flash for the large context window
    answer_res = client_genai.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )

    print(f"ANSWER: {answer_res.text}")
    print("-" * 50)

# NOW TEST THE GLOBAL COUNT
ask_f1_total_history("how many times has mclaren won an F1 race?")

Question: how many times has mclaren won an F1 race?
--------------------------------------------------
ANSWER: Based on the provided data, McLaren has won an F1 race 24 times. Here are the races they won:

*   1973: Australian Grand Prix (Peter Revson)
*   1974: Brazilian Grand Prix (Emerson Fittipaldi)
*   1974: Belgian Grand Prix (Emerson Fittipaldi)
*   1974: Monaco Grand Prix (Emerson Fittipaldi)
*   1974: British Grand Prix (Jody Scheckter)
*   1978: Brazilian Grand Prix (Emerson Fittipaldi)
*   1978: South African Grand Prix (Ronnie Peterson)
*   1978: United States Grand Prix West (Carlos Reutemann)
*   1978: Canadian Grand Prix (Gilles Villeneuve)
*   1978: German Grand Prix (Mario Andretti)
*   1978: Hungarian Grand Prix (Mario Andretti)
*   1978: Belgian Grand Prix (Mario Andretti)
*   1978: Belgian Grand Prix (Mario Andretti)
*   1978: Italian Grand Prix (Niki Lauda)
*   1981: United States Grand Prix West (John Watson)
*   1981: British Grand Prix (John Watson)
*   1982: A

In [233]:
import os
import json
from google import genai
from google.genai import types
from opensearchpy import OpenSearch

# 1. Setup Clients
client_genai = genai.Client(api_key=api_key)
client_os = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False, verify_certs=False, ssl_show_warn=False
)

def ask_f1_intelligent_chat(question):
    print(f"User Question: {question}\n" + "="*60)

    # --- STEP 1: INTENT ROUTING & ENTITY EXTRACTION ---
    # We use Gemini to decide the path and extract the "filter" values
    router_prompt = f"""
    Analyze the following F1 question: "{question}"
    
    Determine if this is:
    1. 'AGGREGATE': A question asking for a total count, career statistic, or list of achievements (e.g., "How many wins...", "Total podiums...").
    2. 'SPECIFIC': A question about a specific race, a single event, or a narrative description (e.g., "What happened at...", "Who crashed...").

    If it is 'AGGREGATE', extract the 'entity_type' (driver/team) and the 'entity_name'.
    
    Return ONLY a JSON object:
    {{"intent": "AGGREGATE" | "SPECIFIC", "entity_type": "driver" | "team" | "none", "entity_name": "string" | "none"}}
    """
    
    route_res = client_genai.models.generate_content(
        model="models/gemini-2.0-flash", 
        contents=router_prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json")
    )
    routing = json.loads(route_res.text)

    # --- STEP 2: PATH EXECUTION ---
    if routing['intent'] == 'AGGREGATE':
        print(f"-> Routing to OpenSearch Aggregations (Target: {routing['entity_name']})")
        
        # We search specifically for instances where the entity is the Winner
        # Using a match_phrase on the entities field we built during indexing
        search_body = {
            "size": 0,
            "query": {
                "bool": {
                    "must": [
                        { "term": { "doc_type": "race_summary" } },
                        { "match_phrase": { "entities": f"Winner: {routing['entity_name']}" } }
                    ]
                }
            },
            "aggs": {
                "total_count": { "value_count": { "field": "metadata.raceId" } }
            }
        }
        
        agg_res = client_os.search(index="hybrid-search-index", body=search_body)
        count = agg_res['aggregations']['total_count']['value']
        
        final_answer = f"Based on the historical records, {routing['entity_name']} has {count} P1 finishes (wins) in the dataset."

    else:
        print("-> Routing to Hybrid Search (RAG)")
        # Standard Hybrid Search for specific stories
        query_res = client_genai.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=3072)
        )
        query_vector = query_res.embeddings[0].values

        search_body = {
            "size": 1,
            "query": {
                "hybrid": {
                    "queries": [
                        { "multi_match": { "query": question, "fields": ["text_representation", "entities^5"] } },
                        { "knn": { "embedding_3072": { "vector": query_vector, "k": 5 } } }
                    ]
                }
            },
            "post_filter": { "term": { "doc_type": "race_summary" } }
        }

        search_res = client_os.search(
            index="hybrid-search-index",
            params={"search_pipeline": "hybrid-search-pipeline"},
            body=search_body
        )

        if search_res['hits']['hits']:
            context = search_res['hits']['hits'][0]['_source']['text_representation']
            
            # Use Gemini to verbalize the answer
            gen_prompt = f"Using this race record: {context}\n\nAnswer this: {question}"
            ans_res = client_genai.models.generate_content(model="gemini-2.0-flash", contents=gen_prompt)
            final_answer = ans_res.text
        else:
            final_answer = "I couldn't find a specific race record matching that query."

    print(f"\nFINAL RESPONSE:\n{final_answer}\n")
    print("="*60)

# --- TEST THE INTELLIGENT ROUTER ---
ask_f1_intelligent_chat("How many times did Massa finish P1 in his career?")
ask_f1_intelligent_chat("What happened to Ferrari in the 2008 Australian Grand Prix?")

User Question: How many times did Massa finish P1 in his career?
-> Routing to OpenSearch Aggregations (Target: Massa)

FINAL RESPONSE:
Based on the historical records, Massa has 11 P1 finishes (wins) in the dataset.

User Question: What happened to Ferrari in the 2008 Australian Grand Prix?
-> Routing to Hybrid Search (RAG)

FINAL RESPONSE:
Based on the provided race record, Ferrari had a very poor showing at the 2008 Australian Grand Prix. Here's what happened:

*   **Kimi Räikkönen:** Retired with an Engine issue.
*   **Felipe Massa:** Retired with an Engine issue.
*   **Rubens Barrichello:** Disqualified.

Essentially, both Ferrari drivers failed to finish the race, and Barrichello didn't even make it to the end due to a disqualification.




In [236]:
ask_f1_intelligent_chat("How many times has Ferrari won the British Grand Prix?")

User Question: How many times has Ferrari won the British Grand Prix?
-> Routing to OpenSearch Aggregations (Target: Ferrari)

FINAL RESPONSE:
Based on the historical records, Ferrari has 0 P1 finishes (wins) in the dataset.



### Generic indexing

The value of one index for all and a standard index is that the data pipeline is standardized for both ingest & hybrid search. 

In [245]:

# ==========================================
# 3. GENERIC DATA PLUMBING (INDEXER)
# ==========================================
def index_data_generically(df, index_name="hybrid-search-index", batch_size=50):
    """Generic indexing engine for any dataset following the 6-column schema."""
    print(f"Indexing {len(df)} docs...")
    
    # Ensure the dataframe has a clean 0-N index to avoid alignment issues
    df = df.reset_index(drop=True)
    
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        texts = batch['text_representation'].tolist()
        
        # Local Embeddings
        v384 = model_384.encode(texts).tolist()
        v1024 = model_1024.encode(texts).tolist()
        
        # Cloud Embeddings
        res_1536 = client_genai.models.embed_content(
            model="gemini-embedding-001", 
            contents=texts, 
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=1536)
        )
        res_3072 = client_genai.models.embed_content(
            model="gemini-embedding-001", 
            contents=texts, 
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=3072)
        )

        actions = []
        # Use range(len(batch)) to ensure we stay within the list bounds of the embeddings
        for idx in range(len(batch)):
            row = batch.iloc[idx]
            
            actions.append({
                "_index": index_name, 
                "_id": row['doc_id'], 
                "doc_type": row['doc_type'], 
                "timestamp": row['timestamp'],
                "text_representation": row['text_representation'], 
                "entities": row['entities'], 
                "metadata": row['metadata'],
                "embedding_384": v384[idx], 
                "embedding_1024": v1024[idx], 
                "embedding_1536": res_1536.embeddings[idx].values, 
                "embedding_3072": res_3072.embeddings[idx].values
            })
        
        helpers.bulk(client_os, actions)
        print(f"Batch {i//batch_size + 1} complete ({len(actions)} docs).")

# ==========================================
# 4. INTELLIGENT ROUTER & CHAT
# ==========================================
def ask_f1_intelligent_chat(question):
    # Intent Routing & Filter Extraction
    extract_prompt = f"""Analyze: "{question}". Return JSON: {{"intent": "AGGREGATE"|"SPECIFIC", "filters": {{"Field": "Value"}}}} Keys: Winner, Winner Team, Race, Year."""
    res = client_genai.models.generate_content(model="gemini-2.0-flash", contents=extract_prompt, config=types.GenerateContentConfig(response_mime_type="application/json"))
    data = json.loads(res.text)

    # Build Dynamic Filter
    must_clauses = [{"term": {"doc_type": "race_summary"}}]
    for k, v in data.get('filters', {}).items():
        must_clauses.append({"match_phrase": {"entities": f"{k}: {v}"}})

    if data['intent'] == 'AGGREGATE':
        body = {"size": 0, "query": {"bool": {"must": must_clauses}}, "aggs": {"total": {"value_count": {"field": "metadata.raceId"}}}}
        agg_res = client_os.search(index="hybrid-search-index", body=body)
        print(f"ANSWER: Found {agg_res['aggregations']['total']['value']} matches for {data['filters']}.")
    else:
        # Standard RAG Path
        query_vec = client_genai.models.embed_content(model="gemini-embedding-001", contents=question, config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=3072)).embeddings[0].values
        body = {"size": 1, "query": {"hybrid": {"queries": [{"multi_match": {"query": question, "fields": ["text_representation", "entities^5"]}}, {"knn": {"embedding_3072": {"vector": query_vec, "k": 5}}}]}}, "post_filter": {"bool": {"must": must_clauses}}}
        search_res = client_os.search(index="hybrid-search-index", params={"search_pipeline": "hybrid-search-pipeline"}, body=body)
        
        if search_res['hits']['hits']:
            ctx = search_res['hits']['hits'][0]['_source']['text_representation']
            ans = client_genai.models.generate_content(model="gemini-2.0-flash", contents=f"Context: {ctx}\nQuestion: {question}")
            print(f"ANSWER: {ans.text}")

# ==========================================
# 5. EXECUTION
# ==========================================
# df_ready = prepare_f1_dataset(df_raw)
# index_data_generically(df_ready)
# ask_f1_intelligent_chat("How many times has Ferrari won the British Grand Prix?")

In [246]:
import os
import json
import time
import pandas as pd
from datetime import datetime
from google import genai
from google.genai import types
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================


client_genai = genai.Client(api_key=api_key)
client_os = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False, verify_certs=False, ssl_show_warn=False
)

# Load local embedding models
model_384 = SentenceTransformer('all-MiniLM-L6-v2')
model_1024 = SentenceTransformer('BAAI/bge-large-en-v1.5')

# F1 Global Status Constants
LAP_DOWN_STATUSES = {"+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps", "+6 Laps", "+7 Laps", "+8 Laps", "+9 Laps", "+10 Laps", "+11 Laps", "+12 Laps", "+13 Laps", "+14 Laps", "+15 Laps", "+16 Laps", "+17 Laps", "+18 Laps", "+19 Laps", "+20 Laps", "+21 Laps", "+22 Laps", "+23 Laps", "+24 Laps", "+25 Laps", "+26 Laps", "+29 Laps", "+30 Laps", "+42 Laps", "+44 Laps", "+46 Laps"}
CRASH_STATUSES = {"Collision", "Accident", "Spun off", "Collision damage", "Debris", "Fatal accident", "Fire"}
MECHANICAL_STATUSES = {"Engine", "Transmission", "Clutch", "Electrical", "Hydraulics", "Gearbox", "Radiator", "Suspension", "Brakes", "Overheating", "Mechanical", "Tyre", "Driver Seat", "Puncture", "Driveshaft", "Fuel pressure", "Front wing", "Water pressure", "Refuelling", "Wheel", "Throttle", "Steering", "Technical", "Electronics", "Broken wing", "Heat shield fire", "Exhaust", "Oil leak", "Wheel rim", "Water leak", "Fuel pump", "Track rod", "Oil pressure", "Pneumatics", "Engine fire", "Tyre puncture", "Wheel nut", "Rear wing", "Fuel system", "Oil line", "Fuel rig", "Launch control", "Drivetrain", "Ignition", "Chassis", "Battery", "Halfshaft", "Crankshaft", "Alternator", "Differential", "Wheel bearing", "Oil pump", "Fuel leak", "Injection", "Distributor", "Turbo", "CV joint", "Water pump", "Spark plugs", "Fuel pipe", "Oil pipe", "Axle", "Water pipe", "Magneto", "Supercharger", "Engine misfire", "ERS", "Power Unit", "Brake duct", "Seat", "Damage", "Cooling system", "Undertray"}
FUEL_STATUSES = {"Out of fuel", "Fuel", "Fuel system", "Fuel pump", "Fuel leak", "Fuel pipe", "Fuel rig", "Fuel pressure"}
ADMIN_STATUSES = {"Disqualified", "Retired", "Withdrew", "Not classified", "107% Rule", "Safety", "Did not qualify", "Did not prequalify", "Excluded", "Not restarted", "Underweight", "Safety belt"}
HEALTH_STATUSES = {"Injured", "Injury", "Driver unwell", "Illness", "Physical", "Eye injury", "Safety concerns"}

# ==========================================
# 2. DOMAIN LOGIC (F1 VERBALIZER)
# ==========================================
def classify_status_group(status):
    status = str(status)
    if status == "Finished": return "finished"
    if status in LAP_DOWN_STATUSES: return "lapped"
    if status in CRASH_STATUSES: return "crash"
    if status in MECHANICAL_STATUSES: return "mechanical"
    if status in FUEL_STATUSES: return "fuel"
    if status in ADMIN_STATUSES: return "admin"
    if status in HEALTH_STATUSES: return "health"
    return "other"

def format_race_date(date_str):
    if pd.isna(date_str) or date_str in (None, "", r'\N'): return None
    try:
        return datetime.strptime(str(date_str), "%Y-%m-%d").strftime("%-d %B %Y")
    except: return str(date_str)

def prepare_f1_dataset(df_raw):
    """Transforms raw F1 results into the standardized 6-column generic schema."""
    def verbalize(group):
        row = group.iloc[0]
        group['status_group'] = group['status'].apply(classify_status_group)
        group['dnf_flag'] = ~group['status_group'].isin(['finished', 'lapped'])
        
        podium = group[group['positionOrder'].isin([1, 2, 3])].sort_values('positionOrder')
        winner_name = podium.iloc[0]['surname'] if not podium.empty else "N/A"
        winner_team = podium.iloc[0]['team_name'] if not podium.empty else "N/A"
        
        dnfs = group[group['dnf_flag']]
        mech_dnfs = dnfs[dnfs['status_group'] == 'mechanical']
        
        # Build the Narrative
        narrative = (f"The {row['year']} {row['name_race']} was held at {row['circuit_name']} on {format_race_date(row['date'])}. "
                     f"Winner: {winner_name} for {winner_team}. Total Retirements: {len(dnfs)}, "
                     f"of which {len(mech_dnfs)} were mechanical issues.")

        # RETURN STANDARDIZED SCHEMA
        return pd.Series({
            'doc_id': f"race_{row['raceId']}",
            'doc_type': "race_summary",
            'timestamp': f"{int(row['year'])}-01-01T00:00:00Z",
            'text_representation': narrative,
            'entities': f"Winner: {winner_name} | Winner Team: {winner_team} | Race: {row['name_race']} | Year: {row['year']}",
            'metadata': {"raceId": int(row['raceId']), "year": int(row['year']), "winner": winner_name, "winner_team": winner_team}
        })

    return df_raw.groupby('raceId').apply(verbalize).reset_index(drop=True)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [247]:
df_ready = prepare_f1_dataset(df)
index_data_generically(df_ready)
ask_f1_intelligent_chat("How many times has Ferrari won the British Grand Prix?")

/var/folders/24/y__mx0xd3rn5g48sf3wlvf080000gn/T/ipykernel_19047/3398564292.py:84: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df_raw.groupby('raceId').apply(verbalize).reset_index(drop=True)


Indexing 1125 docs...
Batch 1 complete (50 docs).
Batch 2 complete (50 docs).
Batch 3 complete (50 docs).
Batch 4 complete (50 docs).
Batch 5 complete (50 docs).
Batch 6 complete (50 docs).
Batch 7 complete (50 docs).
Batch 8 complete (50 docs).
Batch 9 complete (50 docs).
Batch 10 complete (50 docs).
Batch 11 complete (50 docs).
Batch 12 complete (50 docs).
Batch 13 complete (50 docs).
Batch 14 complete (50 docs).
Batch 15 complete (50 docs).
Batch 16 complete (50 docs).
Batch 17 complete (50 docs).
Batch 18 complete (50 docs).
Batch 19 complete (50 docs).
Batch 20 complete (50 docs).
Batch 21 complete (50 docs).
Batch 22 complete (50 docs).
Batch 23 complete (25 docs).
ANSWER: The provided context only states that Ferrari won the 1958 British Grand Prix. It doesn't give any information about other wins. So, based on the given information, Ferrari has won the British Grand Prix at least once. To know the total number of times, further research would be needed.



In [249]:
df_ready[0:5].to_json("sample_f1_race.json", orient="records", lines=True)

In [248]:
import os
import json
from google import genai
from google.genai import types
from opensearchpy import OpenSearch

# 1. SETUP CLIENTS
client_genai = genai.Client(api_key=api_key)
client_os = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False, verify_certs=False, ssl_show_warn=False
)

# ==========================================
# 2. DEFINE LOCAL CALLBACK TOOLS
# ==========================================

def get_f1_stats(winner_team: str = None, winner_name: str = None, race_name: str = None, year: int = None):
    """
    Retrieves the total count of race wins from the database. 
    Use this for any question asking for 'how many', 'total', 'count', or 'career wins'.
    
    Args:
        winner_team: The name of the team (e.g., 'Ferrari', 'McLaren').
        winner_name: The surname of the driver (e.g., 'Massa', 'Schumacher').
        race_name: The name of the Grand Prix (e.g., 'British Grand Prix').
        year: The specific year of the race.
    """
    must_clauses = [{"term": {"doc_type": "race_summary"}}]
    
    # Building filters based on the prefixes we used during indexing
    if winner_team:
        must_clauses.append({"match_phrase": {"entities": f"Winner Team: {winner_team}"}})
    if winner_name:
        must_clauses.append({"match_phrase": {"entities": f"Winner: {winner_name}"}})
    if race_name:
        must_clauses.append({"match_phrase": {"entities": f"Race: {race_name}"}})
    if year:
        must_clauses.append({"term": {"metadata.year": int(year)}})

    query = {
        "size": 0,
        "query": {"bool": {"must": must_clauses}},
        "aggs": {
            "total_wins": {"value_count": {"field": "metadata.raceId"}}
        }
    }
    
    res = client_os.search(index="hybrid-search-index", body=query)
    count = res['aggregations']['total_wins']['value']
    
    # This return value goes back to Gemini to be verbalized
    return {"total_wins": count, "search_criteria": {"team": winner_team, "driver": winner_name, "race": race_name, "year": year}}

def search_f1_narratives(query: str):
    """
    Searches the race database for specific stories, details, or descriptions of what happened.
    Use this for questions like 'What happened at...', 'Who crashed...', or 'Tell me about...'.
    
    Args:
        query: A descriptive search string.
    """
    # 1. Embed the query locally or via API
    emb_res = client_genai.models.embed_content(
        model="gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=3072)
    )
    vector = emb_res.embeddings[0].values

    # 2. Hybrid Search
    search_body = {
        "size": 1,
        "query": {
            "hybrid": {
                "queries": [
                    {"multi_match": {"query": query, "fields": ["text_representation", "entities^5"]}},
                    {"knn": {"embedding_3072": {"vector": vector, "k": 5}}}
                ]
            }
        },
        "post_filter": {"term": {"doc_type": "race_summary"}}
    }

    res = client_os.search(
        index="hybrid-search-index", 
        params={"search_pipeline": "hybrid-search-pipeline"}, 
        body=search_body
    )
    
    if res['hits']['hits']:
        return {"race_detail": res['hits']['hits'][0]['_source']['text_representation']}
    return {"error": "No specific race details found."}

# ==========================================
# 3. INITIALIZE THE AGENT
# ==========================================

# Register the functions as tools
f1_tools = [get_f1_stats, search_f1_narratives]

# Create a chat session with automatic function calling enabled
chat = client_genai.chats.create(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        tools=f1_tools,
        # This allows the SDK to automatically execute the local function
        # and send the result back to Gemini without you doing it manually.
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False)
    )
)

def ask_f1_agent(question):
    print(f"Question: {question}")
    response = chat.send_message(question)
    print(f"AGENT RESPONSE: {response.text}\n")

# ==========================================
# 4. RUN TESTS
# ==========================================

# This should trigger get_f1_stats
ask_f1_agent("How many times did Ferrari win the British Grand Prix?")

# This should trigger search_f1_narratives
ask_f1_agent("What happened to Ferrari in the 2008 Australian Grand Prix?")

Question: How many times did Ferrari win the British Grand Prix?
AGENT RESPONSE: Could you please provide the year and the name of the driver who won the British Grand Prix for Ferrari?

Question: What happened to Ferrari in the 2008 Australian Grand Prix?
AGENT RESPONSE: I've searched the database for information about the 2008 Australian Grand Prix. The race was won by Hamilton for McLaren, and there were a total of 16 retirements, 7 of which were due to mechanical issues. Unfortunately, the database does not provide specific details regarding what happened to Ferrari in that race.

